# Report + Business Stakeholder Review — Agents Review

Human-run notebook for the **agent slice** of Sean Step 6: the new `agents/report_writer.py` and the new `report_review` mode on `agents/business_stakeholder.py`.

Plan: `project_planning/sean_step_artifacts/Report_Business_Review_Implementation_Plan.md`  
Checklist: `project_planning/sean_step_artifacts/Report_Business_Review_Checklist.md`

This notebook stubs the LLM with fake adapters so it can be exercised offline. Run cells top to bottom to:
- exercise `report_writer_node(mode='generate')` on a representative final state
- exercise the missing-`evaluation_result` fallback
- exercise the missing-`modeling_verdict` guardrail
- exercise `business_stakeholder_node(mode='report_review')` for accept / revise_report / revise_modeling outcomes
- confirm the existing `raw_review` / `processed_review` modes still pass

In [ ]:
from pathlib import Path
from pprint import pprint
import subprocess
from typing import Any

from multi_agent_ds.agents import business_stakeholder as biz_agent
from multi_agent_ds.agents import report_writer as report_agent
from multi_agent_ds.agents.business_stakeholder import business_stakeholder_node
from multi_agent_ds.agents.report_writer import report_writer_node

def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find the repo root.")

ROOT = resolve_repo_root()
print("Repo root:", ROOT)

def run_pytest(args: list[str]) -> None:
    cmd = ["uv", "run", "pytest", *args]
    print("Running:", " ".join(cmd))
    completed = subprocess.run(cmd, cwd=ROOT)
    if completed.returncode != 0:
        raise RuntimeError(f"pytest failed with exit code {completed.returncode}")

SETTINGS = {"llm": {"providers": {"openai": {"model": "gpt-4o", "temperature": 0.2, "max_tokens": 1000}}}}

PROMPTS = {
    "report_writer": {
        "system": "You are the report writer.",
        "generate": (
            "{critique_section}DATA: {data_summary}\n\n"
            "MODELING: {modeling_summary}\n\n"
            "EVALUATION: {evaluation_summary}\n\n"
            "TRACE: {decision_trace}"
        ),
    },
    "business_stakeholder": {
        "system": "You are a business stakeholder.",
        "eda_review": "stage={review_stage}; eda={eda_json}",
        "report_review": (
            "REPORT:\n{report_markdown}\n\n"
            "MODELING: {modeling_summary}\n\n"
            "EVALUATION: {evaluation_summary}"
        ),
    },
}

def install_prompts(monkeypatch_target):
    monkeypatch_target.load_prompts_config = lambda: PROMPTS


## 1. Define a fake LLM adapter that captures the rendered prompt

The report writer uses `adapter.chat(...)` (raw markdown). The business stakeholder uses `adapter.structured_output(...)` (validates against `BusinessReviewOutput`). The fake covers both so this notebook never calls the real OpenAI API.

In [ ]:
class FakeChatAdapter:
    last_user_content: str = ""

    def __init__(self, settings):
        self.settings = settings

    def chat(self, messages, tools=None):
        FakeChatAdapter.last_user_content = messages[-1]["content"]
        return {
            "content": (
                "# Stakeholder report\n\n"
                "**What we tested:** a binary classification problem on synthetic credit data.\n"
                "**Models tried:** LightGBM and Logistic Regression.\n"
                "**Winner:** LightGBM (gini 0.66).\n"
                "**Top features:** age, income, credit score.\n"
                "**Confidence:** moderate to high; CV/test gap stable.\n"
                "**Where to dig deeper:** see the experiment log under reports/.\n"
            ),
            "tool_calls": [],
            "usage": {"input_tokens": 100, "output_tokens": 80, "total_tokens": 180},
        }

class FakeBizReviewAdapter:
    next_action: str = "accept"
    last_user_content: str = ""

    def __init__(self, settings):
        self.settings = settings

    def structured_output(self, messages, schema):
        FakeBizReviewAdapter.last_user_content = messages[-1]["content"]
        next_action = FakeBizReviewAdapter.next_action
        approved = next_action == "accept"
        return {
            "parsed": {
                "summary": "Report is acceptable." if approved else "Report needs work.",
                "approved": approved,
                "next_action": next_action,
                "readability_assessment": "Clear language overall.",
                "plausibility_assessment": "Conclusions are plausible.",
                "concerns": (
                    [] if approved else [{"topic": "clarity", "issue": "Add more caveats.", "severity": "medium"}]
                ),
            }
        }

# Patch at module level for this notebook session. The agents call
# build_adapter(settings, agent=..., task=...) — wrap fake instances so
# the call signature matches.
def fake_chat_build_adapter(*_a, **_kw):
    return FakeChatAdapter(SETTINGS)

def fake_biz_build_adapter(*_a, **_kw):
    return FakeBizReviewAdapter(SETTINGS)

report_agent.build_adapter = fake_chat_build_adapter
report_agent.load_prompts_config = lambda: PROMPTS
biz_agent.build_adapter = fake_biz_build_adapter
biz_agent.load_prompts_config = lambda: PROMPTS
print("Fake build_adapter installed for both agents.")

## 2. Build a representative final-state payload

Mirrors what the pipeline would hold after `ml_modeler` finishes its `final_recommendation` phase, with optional `evaluation_result` from Jonathan's pipeline.

In [ ]:
def build_state(include_evaluation: bool = True) -> dict[str, Any]:
    state: dict[str, Any] = {
        "settings": SETTINGS,
        "data": {
            "data_summary": {
                "n_train": 700,
                "n_validation": 150,
                "n_test": 150,
                "n_features": 9,
                "n_numerical": 6,
                "n_categorical": 3,
                "target_rate_train": 0.18,
                "target_rate_test": 0.18,
                "has_ground_truth": True,
            }
        },
        "modeling_verdict": {
            "summary": "LightGBM leads on every metric.",
            "best_algorithm": "lightgbm",
            "ranked_algorithms": ["lightgbm", "logistic_regression"],
            "final_metrics": {
                "lightgbm": {"gini": 0.66, "roc_auc": 0.83},
                "logistic_regression": {"gini": 0.51, "roc_auc": 0.74},
            },
            "justification": "Lead larger than either model's CV std.",
            "next_action": "proceed_to_evaluation",
        },
        "modeling_results": {
            "final_candidates": {
                "lightgbm": {"final_phase": "feature_selection"},
                "logistic_regression": {"final_phase": "baseline"},
            }
        },
        "agent_decisions": [
            {"agent": "ml_modeler", "phase": "baseline", "summary": "Both worth tuning."},
            {"agent": "ml_reviewer", "phase": "final_recommendation_review", "approved": True, "summary": "Verdict holds."},
        ],
    }
    if include_evaluation:
        state["evaluation_result"] = {
            "winner": "lightgbm",
            "primary_metric": "gini",
            "rankings": {"gini": ["lightgbm", "logistic_regression"]},
            "ground_truth_comparison": {"lightgbm": {"mse_vs_true_prob": 0.0012}},
            "shap": {"top_features": ["age", "income", "credit_score"]},
        }
    return state

state = build_state()
print("State keys:", sorted(state.keys()))


## 3. Exercise `report_writer_node(mode='generate')` — happy path

In [ ]:
result = report_writer_node(state)
print("=== rendered prompt the LLM saw ===")
print(FakeChatAdapter.last_user_content)
print("\n=== returned state update ===")
print("experiment_report length:", len(result["experiment_report"]))
print("current_phase:", result["current_phase"])
print("report_iteration:", result["report_iteration"])
print("agent_decisions[-1]:")
pprint(result["agent_decisions"][-1])
print("\n=== experiment_report content ===")
print(result["experiment_report"])


## 4. Missing-`evaluation_result` fallback

When Jonathan's evaluation pipeline isn't producing yet, the writer must still produce a report — with an honest caveat.

In [ ]:
result = report_writer_node(build_state(include_evaluation=False))
print("agent_decisions[-1]:", result["agent_decisions"][-1])
print("\n--- relevant prompt slice ---")
for line in FakeChatAdapter.last_user_content.splitlines():
    if "EVALUATION" in line or "available" in line.lower() or "caveat" in line.lower():
        print(line)


## 5. Missing-`modeling_verdict` guardrail

The writer cannot run before the modeler finishes. The agent raises a clear error rather than producing an empty or made-up report.

In [ ]:
no_verdict_state = build_state()
no_verdict_state.pop("modeling_verdict")
try:
    report_writer_node(no_verdict_state)
except ValueError as exc:
    print("raised ValueError as expected:")
    print("  ", exc)


## 6. Business stakeholder `report_review` — accept

Now hand the writer's output to the business stakeholder for the realism / readability review.

In [ ]:
FakeBizReviewAdapter.next_action = "accept"
state_with_report = build_state()
state_with_report["experiment_report"] = report_writer_node(state_with_report)["experiment_report"]

review = business_stakeholder_node(state_with_report, mode="report_review")
print("business_review:")
pprint(review["business_review"])
print("should_revise_report:", review["should_revise_report"])
print("should_revise_modeling:", review["should_revise_modeling"])
print("current_phase:", review["current_phase"])


## 7. Business stakeholder `report_review` — revise_report

Simulate the reviewer flagging the writeup. The router will now loop back to `report_writer` (under the `report_iteration` cap).

In [ ]:
FakeBizReviewAdapter.next_action = "revise_report"
review = business_stakeholder_node(state_with_report, mode="report_review")
print("business_review:")
pprint(review["business_review"])
print("should_revise_report:", review["should_revise_report"])
print("should_revise_modeling:", review["should_revise_modeling"])


## 8. Business stakeholder `report_review` — revise_modeling

Simulate the reviewer concluding that the underlying modeling result itself is implausible. The router will reopen the modeling loop.

In [ ]:
FakeBizReviewAdapter.next_action = "revise_modeling"
review = business_stakeholder_node(state_with_report, mode="report_review")
print("should_revise_report:", review["should_revise_report"])
print("should_revise_modeling:", review["should_revise_modeling"])
print("agent_decisions[-1]:", review["agent_decisions"][-1])


## 9. Confirm rewrite-loop critique surfacing

On the rewrite, the writer's prompt must include the prior reviewer's concerns so it can answer them in the next draft.

In [ ]:
rewrite_state = build_state()
rewrite_state["current_phase"] = "report_generate"
rewrite_state["report_iteration"] = 1
rewrite_state["business_review"] = {
    "summary": "Tone is too jargon-heavy.",
    "approved": False,
    "next_action": "revise_report",
    "readability_assessment": "Drop technical terms.",
    "plausibility_assessment": "OK",
    "concerns": [{"topic": "tone", "issue": "too technical", "severity": "medium"}],
}
result = report_writer_node(rewrite_state)
print("new report_iteration:", result["report_iteration"])
print("\n--- prompt now contains the critique ---")
print(
    [line for line in FakeChatAdapter.last_user_content.splitlines() if "critique" in line.lower() or "too technical" in line.lower() or "revise" in line.lower()]
)


## 10. Run the agent tests

Includes both the new `report_writer` tests, the new `business_stakeholder` `report_review` tests, and the regression test that the existing `raw_review` mode still works.

In [ ]:
run_pytest([
    "tests/test_report_business_review.py",
    "-k",
    "report_writer or business_stakeholder",
    "-v",
])


## 11. Sign-off

If the agent outputs above are coherent and the tests pass, tick the **Step 2 (report writer agent)** and **Step 3 (business stakeholder report_review mode)** human-review boxes in `Report_Business_Review_Checklist.md`.